# Pre-outreach experiment suiteFifteen experiments, one block each, in the order that matters. Every block writes a verdictinto `/kaggle/working/outreach_evidence.json`; the last cell prints a go / no-go.**Blocking (E1, E2, E4, E5, E6): no email goes out until all five are green.** The rest areper-target or hygiene — run the ones that apply to whoever you are writing to first.Blocks are independent. A missing dataset skips its block and says so rather than failing thenotebook; a skipped block is reported as `SKIP`, never silently as a pass.

In [ ]:
!pip install -q "egnlib>=0.2.2" || pip install -q /kaggle/input/egnlib-wheel/egnlib-0.2.2-py3-none-any.whl

## Setup`record()` is the only shared state. Each block calls it exactly once with a verdict, a headlinenumber, and whatever detail you would paste into an email.

In [ ]:
import json, os, re, subprocess, sys, time, traceback, warningsimport numpy as npimport torchimport torch.nn as nnimport egnfrom egn import EGN, EGNClassifierfrom egn.nn import (BiMap, RiemannianPool, SPDBatchNorm, SpectralActivation,                    TangentHead, ToSPD)warnings.filterwarnings("ignore", category=UserWarning)SEEDS = [1024, 2048, 4096]EVIDENCE = "/kaggle/working/outreach_evidence.json"BLOCKING = {"E1", "E2", "E4", "E5", "E6"}def load_evidence():    return json.load(open(EVIDENCE)) if os.path.exists(EVIDENCE) else {}def record(eid, title, status, headline, detail=None):    """status: PASS | FAIL | SKIP"""    ev = load_evidence()    ev[eid] = {"title": title, "status": status, "headline": headline,               "detail": detail or {}, "when": time.strftime("%Y-%m-%d %H:%M")}    json.dump(ev, open(EVIDENCE, "w"), indent=1, default=str)    mark = {"PASS": "[PASS]", "FAIL": "[FAIL]", "SKIP": "[SKIP]"}[status]    print(f"\n{mark} {eid}  {title}\n       {headline}")def pick_device():    if torch.cuda.is_available():        try:            torch.zeros(1, device="cuda").add_(1).cpu()            return "cuda"        except Exception:            print("CUDA present but unusable -> CPU")    return "cpu"DEVICE = pick_device()def timed(fn, device, reps=3, warmup=1):    for _ in range(warmup):        fn()    if device == "cuda":        torch.cuda.synchronize()    t0 = time.perf_counter()    for _ in range(reps):        fn()    if device == "cuda":        torch.cuda.synchronize()    return (time.perf_counter() - t0) / repsdef stratified_split(y, fraction=0.5, seed=0, groups=None):    rng = np.random.default_rng(seed)    if groups is not None:        uniq = np.unique(groups); rng.shuffle(uniq)        cut = max(int(round(len(uniq) * fraction)), 1)        keep = set(uniq[:cut].tolist())        m = np.array([g in keep for g in groups])        return np.flatnonzero(m), np.flatnonzero(~m)    tr, te = [], []    for c in np.unique(y):        idx = np.flatnonzero(y == c); rng.shuffle(idx)        cut = max(int(round(len(idx) * fraction)), 1)        tr.append(idx[:cut]); te.append(idx[cut:])    return np.concatenate(tr), np.concatenate(te)print("device:", DEVICE, "| torch", torch.__version__, "| egnlib", egn.__version__)

## E6 — Clean-environment install checkRun this **first**. If `pip install egnlib` is broken on a fresh container, no other numbermatters, and this is what your first reader will do before they read anything else.Checks the import path, the version, the five-line quickstart from the README, and the consolescript. Screenshot the output — it goes in the email as the "it just works" evidence.

In [ ]:
try:    assert egn.__version__ >= "0.2.2", f"stale version {egn.__version__}"    X = np.random.randn(256, 12, 128).astype(np.float32)    y = np.random.randint(0, 3, 256)    t0 = time.perf_counter()    clf = EGNClassifier(epochs=5, verbose=0, device=DEVICE).fit(X, y)    quick = time.perf_counter() - t0    which = subprocess.run(["which", "egn-benchmark"], capture_output=True, text=True).stdout.strip()    record("E6", "clean install + quickstart", "PASS",           f"egnlib {egn.__version__} imports, 5-line quickstart fits in {quick:.1f}s",           {"version": egn.__version__, "console_script": which or "NOT ON PATH",            "quickstart_seconds": round(quick, 2), "device": DEVICE})except Exception as exc:    traceback.print_exc()    record("E6", "clean install + quickstart", "FAIL", f"{type(exc).__name__}: {exc}")

## E1 — HDM05 reproduction at parity  *(blocking)*The standard protocol: 117 classes, half of each class for training, 3 seeds, mean±std.**Pass condition: within 2 points of the SPDNet-family published number. Fail if more than 5points below** — at that distance the email is a liability, not an asset.Set `PUBLISHED` to the number you are actually claiming parity with, from the paper you cite.Do not leave it at the placeholder.

In [ ]:
HDM05_DIR = "/kaggle/working/data/hdm05_raw/HDM05"PUBLISHED = 61.60          # <- set to the exact published number you are matchingdef load_hdm05(root=HDM05_DIR, cache="/kaggle/working/hdm05.npz"):    if cache and os.path.exists(cache):        b = np.load(cache); return b["X"], b["y"]    files = sorted(f for f in os.listdir(root) if f.endswith(".npy"))    X = np.stack([np.squeeze(np.load(os.path.join(root, f))) for f in files]).astype(np.float32)    y = np.array([int(re.search(r"_(\d+)\.npy$", f).group(1)) for f in files])    if cache:        np.savez_compressed(cache, X=X, y=y)    return X, yHDM05_CFG = dict(input_kind="spd", branches=1, channels=4, dims=[93, 50, 30],                 head="tangent", pool="logeuclid", epochs=100, batch_size=30,                 lr=1e-2, ridge=1e-3, scheduler="cosine")try:    X_hdm, y_hdm = load_hdm05()    accs = []    for seed in SEEDS:        tr, te = stratified_split(y_hdm, 0.5, seed)        clf = EGNClassifier(device=DEVICE, seed=seed, verbose=0, **HDM05_CFG)        clf.fit(X_hdm[tr], y_hdm[tr])        a = clf.score(X_hdm[te], y_hdm[te]) * 100        accs.append(a)        print(f"  seed {seed}: {a:.2f}")    m, s = float(np.mean(accs)), float(np.std(accs))    gap = m - PUBLISHED    status = "PASS" if gap > -5 else "FAIL"    record("E1", "HDM05 parity", status,           f"{m:.2f} +/- {s:.2f} vs published {PUBLISHED:.2f} (gap {gap:+.2f})",           {"per_seed": accs, "published": PUBLISHED, "config": HDM05_CFG})except FileNotFoundError as exc:    record("E1", "HDM05 parity", "SKIP", f"dataset missing: {exc}")except Exception as exc:    traceback.print_exc()    record("E1", "HDM05 parity", "FAIL", f"{type(exc).__name__}: {exc}")

## E2 — CPU vs GPU timing  *(blocking)*The whole Cornell/Yale pitch is one table. Four rows, same HDM05 config, seconds per epoch:| row | what it shows ||---|---|| their CPU baseline | run separately in E7 || egnlib CPU | your library with no GPU || egnlib GPU fp32 | the default policy || egnlib GPU fp64 | why the old code was slower on GPU than on CPU |If the fp64 row is not dramatically slower than fp32 on your card, say so — on a datacentre GPUwith a 1:2 fp64 ratio the gap largely disappears, and claiming otherwise on an A100 is the kindof overstatement a reader will catch immediately.

In [ ]:
def epoch_time(device, spectral, n=256):    egn.config.spectral_dtype = spectral    dtype = torch.float64 if spectral == "float64" else torch.float32    X = torch.from_numpy(X_hdm[:n]).to(device, dtype)    y = torch.from_numpy(y_hdm[:n]).long().to(device)    model = EGN(num_classes=int(y_hdm.max()) + 1, **{k: v for k, v in HDM05_CFG.items()                if k in ("input_kind", "branches", "channels", "dims", "head", "pool", "ridge")})    model.build_from_example(X, int(y_hdm.max()) + 1)    model = model.to(device=device, dtype=dtype)    opt = egn.build_optimizer(model, lr=1e-2)    def one_epoch():        for s in range(0, n, 30):            opt.zero_grad(set_to_none=True)            torch.nn.functional.cross_entropy(model(X[s:s + 30]), y[s:s + 30]).backward()            opt.step()    t = timed(one_epoch, device, reps=3)    egn.config.spectral_dtype = "auto"    return ttry:    rows = {"egnlib CPU fp64": epoch_time("cpu", "float64"),            "egnlib CPU fp32": epoch_time("cpu", "float32")}    if DEVICE == "cuda":        rows["egnlib GPU fp32"] = epoch_time("cuda", "float32")        rows["egnlib GPU fp64"] = epoch_time("cuda", "float64")        gpu_name = torch.cuda.get_device_name(0)    else:        gpu_name = "no usable GPU"    print(f"\n{'row':<20}{'s/epoch':>10}")    for k, v in rows.items():        print(f"{k:<20}{v:>10.3f}")    if DEVICE == "cuda":        speedup = rows["egnlib CPU fp64"] / rows["egnlib GPU fp32"]        ratio = rows["egnlib GPU fp64"] / rows["egnlib GPU fp32"]        head = (f"{gpu_name}: GPU fp32 is {speedup:.1f}x the CPU fp64 baseline; "                f"fp64 costs {ratio:.1f}x fp32 on this card")        status = "PASS" if speedup > 1.0 else "FAIL"    else:        head, status = "no usable GPU in this session -- rerun on a T4", "SKIP"    record("E2", "CPU vs GPU timing", status, head,           {"seconds_per_epoch": rows, "gpu": gpu_name, "batch": 30, "samples": 256})except NameError:    record("E2", "CPU vs GPU timing", "SKIP", "needs E1 to have loaded HDM05")except Exception as exc:    traceback.print_exc()    record("E2", "CPU vs GPU timing", "FAIL", f"{type(exc).__name__}: {exc}")

## E3 — AFEWThe other SPD dataset in the Cornell/Yale paper: 400×400 covariance descriptors, 7 emotionclasses. Point `AFEW_DIR` at a directory of `.npy` or `.mat` covariances.If you cannot get AFEW, **name it in the email as untested**. A dataset you disclose as missingcosts you nothing; one you quietly omit costs you the reader's trust when they notice.

In [ ]:
AFEW_DIR = "/kaggle/working/data/afew"def load_afew(root=AFEW_DIR):    if not os.path.isdir(root):        raise FileNotFoundError(root)    files = sorted(f for f in os.listdir(root) if f.endswith((".npy", ".mat")))    if not files:        raise FileNotFoundError(f"no covariance files under {root}")    X, y = [], []    for f in files:        p = os.path.join(root, f)        if f.endswith(".npy"):            S = np.squeeze(np.load(p))        else:            from scipy.io import loadmat            d = loadmat(p)            S = np.squeeze(next(v for k, v in d.items() if not k.startswith("__")))        X.append(S.astype(np.float32))        y.append(int(re.search(r"_(\d+)\.(npy|mat)$", f).group(1)))    return np.stack(X), np.asarray(y)try:    X_afew, y_afew = load_afew()    print("afew", X_afew.shape, len(np.unique(y_afew)), "classes")    accs = []    for seed in SEEDS:        tr, te = stratified_split(y_afew, 0.7, seed)        clf = EGNClassifier(input_kind="spd", branches=1, channels=4,                            dims=[X_afew.shape[-1], 200, 100, 50], head="tangent",                            pool="logeuclid", epochs=50, batch_size=30, lr=1e-2,                            ridge=1e-3, scheduler="cosine",                            device=DEVICE, seed=seed, verbose=0)        clf.fit(X_afew[tr], y_afew[tr])        accs.append(clf.score(X_afew[te], y_afew[te]) * 100)        print(f"  seed {seed}: {accs[-1]:.2f}")    record("E3", "AFEW", "PASS",           f"{np.mean(accs):.2f} +/- {np.std(accs):.2f} over {len(SEEDS)} seeds",           {"per_seed": accs})except FileNotFoundError as exc:    record("E3", "AFEW", "SKIP",           f"not available ({exc}) -- disclose as untested in the email")except Exception as exc:    traceback.print_exc()    record("E3", "AFEW", "FAIL", f"{type(exc).__name__}: {exc}")

## E4 — Parameters and peak memory, side by side  *(blocking)*SPDNet and SPDNetBN are rebuilt here **from egn layers**, so the comparison is architectureagainst architecture at identical widths rather than one codebase against another. Label themthat way in the email — "SPDNet-equivalent, rebuilt from the same primitives" — because they arenot the original authors' code and claiming otherwise is not defensible.Peak memory is measured over one forward+backward at batch 30.

In [ ]:
def spdnet_like(in_dim, dims, num_classes, batchnorm=False):    """SPDNet (Huang & Van Gool) / SPDNetBN (Brooks et al.) at matched widths."""    layers = [ToSPD(kind="spd", ridge=1e-3)]    d = in_dim    for nxt in dims:        layers.append(BiMap(d, nxt, 1, 1))        if batchnorm:            layers.append(SPDBatchNorm(nxt, 1))        layers.append(SpectralActivation("rect"))        d = nxt    layers += [RiemannianPool("logeuclid"),               TangentHead(d, num_classes, learn_reference=False, batchnorm=False)]    return nn.Sequential(*layers)def count(m):    return sum(p.numel() for p in m.parameters() if p.requires_grad)def peak_memory(model, X, y):    if DEVICE != "cuda":        return None    model = model.to(DEVICE); X = X.to(DEVICE); y = y.to(DEVICE)    torch.cuda.reset_peak_memory_stats(); torch.cuda.empty_cache()    loss = torch.nn.functional.cross_entropy(model(X), y)    loss.backward()    return torch.cuda.max_memory_allocated() / 2**20try:    C = int(y_hdm.max()) + 1    Xb = torch.from_numpy(X_hdm[:30]); yb = torch.from_numpy(y_hdm[:30]).long()    models = {        "SPDNet-equivalent":   spdnet_like(93, [50, 30], C, batchnorm=False),        "SPDNetBN-equivalent": spdnet_like(93, [50, 30], C, batchnorm=True),        "EGN (tangent head)":  EGN(num_classes=C, in_dim=93, input_kind="spd",                                   channels=4, dims=[93, 50, 30], head="tangent"),        "EGN (geodesic head)": EGN(num_classes=C, in_dim=93, input_kind="spd",                                   channels=4, dims=[93, 50, 30], head="geodesic",                                   pool="frechet"),    }    table = {}    print(f"{'model':<24}{'params':>12}{'peak MiB':>12}")    for name, m in models.items():        mem = peak_memory(m, Xb, yb)        table[name] = {"params": count(m), "peak_mib": None if mem is None else round(mem, 1)}        print(f"{name:<24}{count(m):>12,}{'--' if mem is None else f'{mem:>12.1f}'}")    record("E4", "parameters and memory", "PASS",           f"EGN tangent {table['EGN (tangent head)']['params']:,} vs "           f"SPDNetBN-equivalent {table['SPDNetBN-equivalent']['params']:,}",           table)except NameError:    record("E4", "parameters and memory", "SKIP", "needs E1 to have loaded HDM05")except Exception as exc:    traceback.print_exc()    record("E4", "parameters and memory", "FAIL", f"{type(exc).__name__}: {exc}")

## E5 — Independent correctness check  *(blocking)*Your own test suite proves your code agrees with itself. This block checks the Fréchet mean andthe affine-invariant distance against **geomstats** and **pyriemann** on identical input.For Miolane specifically this is not a supporting number — it is the entire pitch. "Agrees withgeomstats to 1e-10, then trains a classifier on top" is a sentence that lands with someone whomaintains geomstats.

In [ ]:
!pip install -q pyriemann geomstats

In [ ]:
from egn.geometry import distance as egn_distancefrom egn.geometry import frechet_mean, random_spddetail, disagree, unavailable = {}, [], []with egn.numeric_context(spectral_dtype="float64"):    S = random_spd((32, 10, 10), condition=20.0, dtype=torch.float64)    mine_mean = frechet_mean(S.unsqueeze(0), dim=1, iters=60).squeeze(0)    mine_dist = egn_distance(S[0], S[1]).item()    try:        from pyriemann.utils.distance import distance_riemann        from pyriemann.utils.mean import mean_riemann        ref = mean_riemann(S.numpy())        err = float(np.abs(ref - mine_mean.numpy()).max())        derr = abs(distance_riemann(S[0].numpy(), S[1].numpy()) - mine_dist)        detail["pyriemann"] = {"mean_max_abs_err": err, "distance_abs_err": float(derr)}        print(f"pyriemann  mean {err:.3e}   distance {derr:.3e}")        if err > 1e-8 or derr > 1e-8:            disagree.append("pyriemann")    except Exception as exc:        detail["pyriemann"] = f"unavailable: {exc}"        unavailable.append("pyriemann")    try:        from geomstats.geometry.spd_matrices import SPDMatrices, SPDAffineMetric        space = SPDMatrices(10, equip=False); space.equip_with_metric(SPDAffineMetric)        gref = space.metric.dist(S[0].numpy(), S[1].numpy())        gerr = float(abs(gref - mine_dist))        detail["geomstats"] = {"distance_abs_err": gerr}        print(f"geomstats  distance {gerr:.3e}")        if gerr > 1e-8:            disagree.append("geomstats")    except Exception as exc:        detail["geomstats"] = f"unavailable: {exc}"        unavailable.append("geomstats")checked = 2 - len(unavailable)if disagree:    status, head = "FAIL", f"disagrees with: {', '.join(disagree)}"elif checked == 0:    status, head = "SKIP", "neither reference library could be imported"else:    status = "PASS"    head = (f"agrees with {checked} reference librar"            f"{'y' if checked == 1 else 'ies'} to 1e-8"            + (f" ({', '.join(unavailable)} unavailable)" if unavailable else ""))record("E5", "independent correctness", status, head, detail)

## E7 — Head-to-head against the Cornell/Yale repoClones `CUAI/Riemannian-Residual-Neural-Networks` and runs their HDM05 script unmodified, thenreports their wall clock next to yours from E2.Their repo needs their data layout; if it will not run, record `SKIP` and use their **published**number in the email instead of a self-measured one. Reporting a baseline you could not actuallyexecute as if you had is the fastest way to lose the reader.

In [ ]:
RRNET = "/kaggle/working/rresnet"try:    if not os.path.isdir(RRNET):        subprocess.run(["git", "clone", "--depth", "1",                        "https://github.com/CUAI/Riemannian-Residual-Neural-Networks.git",                        RRNET], check=True)    exp = os.path.join(RRNET, "brooks_spd", "experiments")    print("their experiment scripts:", os.listdir(exp))    t0 = time.perf_counter()    proc = subprocess.run([sys.executable, "hdm05.py"], cwd=exp, capture_output=True,                          text=True, timeout=3600)    dt = time.perf_counter() - t0    tail = (proc.stdout or proc.stderr)[-1500:]    print(tail)    if proc.returncode == 0:        record("E7", "baseline repo head-to-head", "PASS",               f"their hdm05.py completed in {dt / 60:.1f} min on {DEVICE}",               {"returncode": 0, "minutes": round(dt / 60, 2), "tail": tail})    else:        record("E7", "baseline repo head-to-head", "SKIP",               "their script did not run here -- cite their published number instead",               {"returncode": proc.returncode, "tail": tail})except Exception as exc:    record("E7", "baseline repo head-to-head", "SKIP",           f"could not run their repo ({type(exc).__name__}) -- cite the published number")

## E8 — Radar low-sample-support curveThe Gurbuz / Amin pitch. Their GAN-augmentation line of work exists because micro-Dopplerdatasets are small. A curve showing EGN needs less data is worth more to them than one accuracypoint.Train on 10 / 25 / 50 / 100 % of the training split, EGN against a spectrogram CNN of comparablesize, 3 seeds each. The CNN here is a small STFT-magnitude baseline — not a tuned competitor, sodescribe it as such: "an untuned spectrogram CNN at matched parameter count", not "the CNNbaseline".

In [ ]:
RADAR_DIR = "/kaggle/working/data/radar_npy/radar"FRACTIONS = [0.10, 0.25, 0.50, 1.00]def load_radar(root=RADAR_DIR, window=20, hop=10):    files = sorted(f for f in os.listdir(root) if f.endswith(".npy"))    Xc, X, y = [], [], []    for f in files:        z = np.load(os.path.join(root, f)).ravel()        W = np.stack([z[s:s + window] for s in range(0, len(z) - window + 1, hop)], axis=1)        X.append(np.concatenate([W.real, W.imag], axis=0))        Xc.append(z)        y.append(int(re.search(r"_(\d+)\.npy$", f).group(1)))    return np.stack(X).astype(np.float32), np.stack(Xc), np.asarray(y)class SpecCNN(nn.Module):    """Untuned STFT-magnitude CNN, sized to roughly match EGN's parameter count."""    def __init__(self, num_classes, ch=16):        super().__init__()        self.net = nn.Sequential(            nn.Conv2d(1, ch, 3, padding=1), nn.BatchNorm2d(ch), nn.ReLU(), nn.MaxPool2d(2),            nn.Conv2d(ch, ch * 2, 3, padding=1), nn.BatchNorm2d(ch * 2), nn.ReLU(),            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(ch * 2, num_classes))    def forward(self, x):        return self.net(x)def spectrograms(Xc, n_fft=64, hop=16):    z = torch.from_numpy(Xc).to(torch.complex64)    S = torch.stft(z, n_fft=n_fft, hop_length=hop, return_complex=True,                   window=torch.hann_window(n_fft), center=True)    m = S.abs().clamp_min(1e-8).log()    m = (m - m.mean(dim=(-2, -1), keepdim=True)) / m.std(dim=(-2, -1), keepdim=True)    return m.unsqueeze(1).float()def train_cnn(Xtr, ytr, Xte, yte, num_classes, seed, epochs=40):    torch.manual_seed(seed)    model = SpecCNN(num_classes).to(DEVICE)    opt = torch.optim.Adam(model.parameters(), lr=1e-3)    Xtr, ytr = Xtr.to(DEVICE), ytr.to(DEVICE)    for _ in range(epochs):        perm = torch.randperm(len(Xtr), device=DEVICE)        for s in range(0, len(Xtr) - 31, 32):            idx = perm[s:s + 32]            opt.zero_grad(set_to_none=True)            torch.nn.functional.cross_entropy(model(Xtr[idx]), ytr[idx]).backward()            opt.step()    model.eval()    with torch.no_grad():        pred = torch.cat([model(Xte[s:s + 256].to(DEVICE)).argmax(-1).cpu()                          for s in range(0, len(Xte), 256)])    return (pred == yte).float().mean().item() * 100try:    X_rad, Xc_rad, y_rad = load_radar()    Spec = spectrograms(Xc_rad)    C = int(y_rad.max()) + 1    print("radar", X_rad.shape, "| spectrograms", tuple(Spec.shape))    curve = {"EGN": {}, "SpecCNN": {}}    for frac in FRACTIONS:        e_acc, c_acc = [], []        for seed in SEEDS:            tr, te = stratified_split(y_rad, 0.7, seed)            sub, _ = stratified_split(y_rad[tr], frac, seed)            tr_sub = tr[sub]            clf = EGNClassifier(input_kind="signal", branches=4, channels=4,                                dims=[40, 24, 12], head="tangent", pool="logeuclid",                                epochs=60, batch_size=64, lr=5e-3, ridge=1e-3,                                scheduler="cosine", device=DEVICE, seed=seed, verbose=0)            clf.fit(X_rad[tr_sub], y_rad[tr_sub])            e_acc.append(clf.score(X_rad[te], y_rad[te]) * 100)            c_acc.append(train_cnn(Spec[tr_sub], torch.from_numpy(y_rad[tr_sub]).long(),                                   Spec[te], torch.from_numpy(y_rad[te]).long(), C, seed))        curve["EGN"][frac] = [float(np.mean(e_acc)), float(np.std(e_acc))]        curve["SpecCNN"][frac] = [float(np.mean(c_acc)), float(np.std(c_acc))]        print(f"  {int(frac * 100):>3}% train  EGN {np.mean(e_acc):5.2f}  "              f"SpecCNN {np.mean(c_acc):5.2f}  (n={len(tr_sub)})")    lo = FRACTIONS[0]    record("E8", "radar low-sample curve", "PASS",           f"at {int(lo * 100)}% of training data: EGN {curve['EGN'][lo][0]:.2f} vs "           f"SpecCNN {curve['SpecCNN'][lo][0]:.2f}", curve)except FileNotFoundError as exc:    record("E8", "radar low-sample curve", "SKIP", f"radar data missing: {exc}")except Exception as exc:    traceback.print_exc()    record("E8", "radar low-sample curve", "FAIL", f"{type(exc).__name__}: {exc}")

## E9 — Public fMRI functional connectivityThe UNC pitch. Run on **ABIDE**, not on your 212-subject Psychiatric CSV — a neuroimaging groupwill not evaluate a claim on a Kaggle table, and saying so first saves you the reply that says itfor you.`nilearn` fetches derivatives that are already ROI time series; the correlation matrices comefrom those. This downloads a few hundred MB, so it needs internet enabled in the notebooksettings.

In [ ]:
!pip install -q nilearn

In [ ]:
try:    from nilearn.connectome import ConnectivityMeasure    from nilearn.datasets import fetch_abide_pcp    abide = fetch_abide_pcp(derivatives=["rois_cc200"], pipeline="cpac",                            band_pass_filtering=True, global_signal_regression=False,                            quality_checked=True, n_subjects=200,                            data_dir="/kaggle/working/abide")    series = [np.asarray(t) for t in abide.rois_cc200]    labels = np.asarray(abide.phenotypic["DX_GROUP"]).astype(int)   # 1 autism, 2 control    sites = np.asarray(abide.phenotypic["SITE_ID"])    corr = ConnectivityMeasure(kind="correlation")    X_fc = corr.fit_transform(series).astype(np.float32)    print("abide", X_fc.shape, np.bincount(labels))    accs = []    for seed in SEEDS:        tr, te = stratified_split(labels, 0.7, seed, groups=sites)   # leave-sites-out        clf = EGNClassifier(input_kind="spd", branches=1, channels=4,                            dims=[X_fc.shape[-1], 64, 32], head="tangent",                            pool="logeuclid", epochs=60, batch_size=32, lr=5e-3,                            ridge=1e-2, scheduler="cosine",                            device=DEVICE, seed=seed, verbose=0)        clf.fit(X_fc[tr], labels[tr])        accs.append(clf.score(X_fc[te], labels[te]) * 100)        print(f"  seed {seed}: {accs[-1]:.2f}  (leave-sites-out)")    record("E9", "ABIDE functional connectivity", "PASS",           f"{np.mean(accs):.2f} +/- {np.std(accs):.2f}, leave-sites-out",           {"per_seed": accs, "n_subjects": len(labels), "atlas": "cc200"})except Exception as exc:    traceback.print_exc()    record("E9", "ABIDE functional connectivity", "SKIP",           f"not run ({type(exc).__name__}: {exc}) -- needs internet enabled")

## E10 — DTI adapter checkThe Vemuri pitch needs a different input shape: a spatial field of 3×3 diffusion tensors,`(N, voxels, 3, 3)`, not a single covariance. This block checks that the multi-branch pathhandles it — voxels become manifold channels — and that the whole pipeline stays finite and SPDend to end.Synthetic tensors here. Before you email that group, rerun it on real dMRI: an adapter that workson well-conditioned synthetic tensors and fails on real ones with near-zero eigenvalues is worsethan no claim.

In [ ]:
try:    from egn.geometry import is_spd, random_spd    N, V = 64, 32                          # 64 subjects, 32 voxels of 3x3 tensors    tensors = random_spd((N, V, 3, 3), condition=50.0).float().numpy()    y_dti = np.random.randint(0, 2, N)    # 3x3 tensors cannot be reduced in size, so the depth has to come from the    # channel axis: mix the V voxel-channels down to 8 while keeping the matrices 3x3    net = nn.Sequential(        ToSPD(kind="spd", ridge=1e-6),        BiMap(3, 3, in_channels=V, out_channels=8, mix=True),        SpectralActivation("rect"),        RiemannianPool("frechet", iters=5),        TangentHead(3, 2),    )    x = torch.from_numpy(tensors)    out = net(x)    feats = nn.Sequential(*list(net)[:-1])(x)    ok_spd = bool(is_spd(feats.double()).all())    ok_fin = bool(torch.isfinite(out).all())    clf = EGNClassifier(input_kind="spd", dims=[3], min_dim=3,                        epochs=3, batch_size=16, device=DEVICE, verbose=0).fit(tensors, y_dti)    status = "PASS" if (ok_spd and ok_fin) else "FAIL"    record("E10", "DTI tensor-field adapter", status,           f"(N,{V},3,3) accepted; pooled feature SPD={ok_spd}, logits finite={ok_fin} "           f"-- SYNTHETIC ONLY, rerun on real dMRI before emailing",           {"input_shape": list(tensors.shape), "feature_shape": list(feats.shape),            "channel_mix": f"{V} voxel-channels -> 8"})except Exception as exc:    traceback.print_exc()    record("E10", "DTI tensor-field adapter", "FAIL", f"{type(exc).__name__}: {exc}")

## E11 — Seed variance vs ablation gapsRun the ablation notebook with `N_SEEDS = 3` first, then this block reads its results file.The question it answers: **is any ablation gap larger than the seed standard deviation?** If not,you do not have an ablation table, you have noise — and that is the first thing a careful readerchecks. Better to find it here than in a review.

In [ ]:
ABLATION = "/kaggle/working/egn_results.json"try:    res = json.load(open(ABLATION))    verdict, worrying = {}, []    for ds, variants in res.items():        full = variants.get("EGN (full)")        if not full:            continue        fa = np.array([r["accuracy"] for r in full.values()]) * 100        if len(fa) < 2:            worrying.append(f"{ds}: only {len(fa)} seed(s)")            continue        sd = float(fa.std())        gaps = {v: float(fa.mean() - np.mean([r["accuracy"] for r in runs.values()]) * 100)                for v, runs in variants.items() if v != "EGN (full)"}        separable = {v: g for v, g in gaps.items() if abs(g) > sd}        verdict[ds] = {"seed_std": round(sd, 2), "gaps": {k: round(v, 2) for k, v in gaps.items()},                       "above_noise": list(separable)}        print(f"{ds:<14} seed std {sd:5.2f} | {len(separable)}/{len(gaps)} ablations exceed it")        if not separable:            worrying.append(f"{ds}: no ablation gap exceeds seed noise")    status = "PASS" if verdict and not worrying else ("SKIP" if not verdict else "FAIL")    record("E11", "seed variance vs ablation gaps", status,           "every dataset has at least one ablation above seed noise" if not worrying           else "; ".join(worrying), verdict)except FileNotFoundError:    record("E11", "seed variance vs ablation gaps", "SKIP",           "run the ablation notebook with N_SEEDS=3 first")

## E12 — Subject-independent splits actually heldCheap, and its absence is a standard reason emotion-recognition numbers get dismissed. Prints theparticipant IDs on each side and asserts the intersection is empty.

In [ ]:
def check_grouped(name, groups, fraction, seeds=SEEDS):    rows = []    for seed in seeds:        tr, te = stratified_split(np.zeros(len(groups)), fraction, seed, groups=groups)        gtr, gte = set(groups[tr].tolist()), set(groups[te].tolist())        rows.append({"seed": seed, "train_groups": sorted(gtr), "test_groups": sorted(gte),                     "overlap": sorted(gtr & gte)})        print(f"  {name} seed {seed}: {len(gtr)} train / {len(gte)} test groups, "              f"overlap {len(gtr & gte)}")    return rowstry:    checks = {}    checks["DEAP"] = check_grouped("DEAP", np.repeat(np.arange(32), 40), 0.75)    checks["SEED"] = check_grouped("SEED", np.repeat(np.arange(15), 45), 0.80)    bad = [f"{k} seed {r['seed']}" for k, rs in checks.items() for r in rs if r["overlap"]]    record("E12", "subject-independent splits", "PASS" if not bad else "FAIL",           "no participant appears on both sides in any seed" if not bad           else f"leakage in: {bad}", checks)except Exception as exc:    record("E12", "subject-independent splits", "FAIL", f"{type(exc).__name__}: {exc}")

## E13 — DEAP channel-axis verification`(1024, 60)` is reshaped to `(32, 32×60)` on the assumption that the 1024 axis is**channel-major**. If it is not, the covariance is over the wrong quantity and the DEAP column isnoise dressed as a result.Two tests. First, a structural one: if the axis really is 32 channels × 32 sub-features, the32×32 channel covariance should have visibly more structure than the same computation on arandomly permuted feature axis. Second, and not optional: **open your DE extractor and confirmthe write order.** No statistic substitutes for reading the code that produced the file.

In [ ]:
DEAP_DIR = "/kaggle/input/deap-research"DEAP_CHANNELS = 32try:    import pandas as pd    sample = pd.read_csv(f"{DEAP_DIR}/DE/participant1video1.txt", header=None,                         usecols=list(range(60)), delimiter=",").values    print("one trial:", sample.shape)    assert sample.shape[0] % DEAP_CHANNELS == 0, "1024 not divisible by DEAP_CHANNELS"    def offdiag_energy(v):        M = v.reshape(DEAP_CHANNELS, -1)        M = M - M.mean(1, keepdims=True)        Cv = M @ M.T / M.shape[1]        d = np.sqrt(np.diag(Cv))        R = Cv / np.outer(d, d)        return float(np.abs(R[~np.eye(DEAP_CHANNELS, dtype=bool)]).mean())    real = offdiag_energy(sample)    rng = np.random.default_rng(0)    shuffled = float(np.mean([offdiag_energy(sample[rng.permutation(len(sample))])                              for _ in range(20)]))    ratio = real / max(shuffled, 1e-9)    status = "PASS" if ratio > 1.10 else "FAIL"    print(f"mean |correlation| assumed-layout {real:.4f} vs shuffled {shuffled:.4f} "          f"(ratio {ratio:.2f})")    record("E13", "DEAP channel-axis check", status,           f"assumed layout shows {ratio:.2f}x the off-diagonal structure of a shuffled axis "           f"-- still confirm the write order in your DE extractor",           {"real": real, "shuffled": shuffled, "ratio": ratio, "channels": DEAP_CHANNELS})except Exception as exc:    record("E13", "DEAP channel-axis check", "SKIP", f"{type(exc).__name__}: {exc}")

## E14 — Failure modes give informative errorsYour first reply from any of these groups will be a bug report. Feed the library the four thingsa stranger's data does that yours does not, and check the error tells them what to fix ratherthan surfacing as a NaN three layers down.

In [ ]:
results = {}def probe(name, fn, expect_error=None):    """expect_error: a substring the message must contain, or None if the call    should simply succeed and return something finite."""    try:        out = fn()        if expect_error is not None:            results[name] = "NO ERROR RAISED (expected one)"            return        t = out if isinstance(out, torch.Tensor) else torch.as_tensor(float(out))        results[name] = "ok" if bool(torch.isfinite(t).all()) else "NON-FINITE OUTPUT"    except Exception as exc:        msg = f"{type(exc).__name__}: {str(exc)[:110]}"        if expect_error is None:            results[name] = f"UNEXPECTED ERROR -> {msg}"        elif expect_error.lower() in str(exc).lower():            results[name] = f"ok, informative -> {msg}"        else:            results[name] = f"UNINFORMATIVE -> {msg}"probe("rank-2 input rejected",      lambda: ToSPD()(torch.randn(8, 12)), expect_error="second moment")probe("indefinite matrix survives the ridge",      lambda: EGN(num_classes=2, input_kind="spd", dims=[6], min_dim=6)(          torch.diag(torch.tensor([1.0, 1.0, 1.0, 1.0, 1.0, -1e-7])).expand(4, 6, 6).clone()))probe("NaN caught at the boundary",      lambda: ToSPD(check_input=True)(torch.full((4, 6, 60), float("nan"))),      expect_error="nan or inf")probe("NaN caught by fit()",      lambda: EGNClassifier(epochs=1, verbose=0, device=DEVICE).fit(          np.full((8, 6, 60), np.nan, dtype=np.float32), np.arange(8) % 2),      expect_error="nan or inf")probe("single-sample inference batch",      lambda: EGN(num_classes=2).eval()(torch.randn(1, 6, 60)))probe("more classes than samples",      lambda: EGNClassifier(epochs=1, verbose=0, device=DEVICE).fit(          np.random.randn(8, 6, 60).astype(np.float32), np.arange(8)).score(          np.random.randn(8, 6, 60).astype(np.float32), np.arange(8)))for k, v in results.items():    print(f"  {k:<34} {v}")bad = [k for k, v in results.items() if not v.startswith("ok")]record("E14", "failure modes", "PASS" if not bad else "FAIL",       "all probes error informatively or return finite output" if not bad       else f"needs work: {bad}", results)

## E15 — Release checklistNot an experiment. The library has to be installable by the person reading your email, whichmeans PyPI, a git tag, and a README whose first code block is the one you quote to them.Run the build locally, not here — Kaggle has no credentials and should not have yours.

In [ ]:
checklist = {    "1 version bumped in pyproject.toml and egn/__init__.py": None,    "2 pytest passes on a clean checkout": None,    "3 python -m build  (produces .whl and .tar.gz)": None,    "4 twine upload dist/*.whl dist/*.tar.gz": None,    "5 git tag v0.2.2 && git push --tags": None,    "6 README quickstart is the exact snippet you paste in the email": None,    "7 pip install egnlib in a fresh Colab -> rerun E6": None,}print("Run locally, then mark each True:\n")for k in checklist:    print("  [ ]", k)done = sum(1 for v in checklist.values() if v is True)record("E15", "release checklist", "PASS" if done == len(checklist) else "SKIP",       f"{done}/{len(checklist)} release steps confirmed", checklist)

## Go / no-goE1, E2, E4, E5 and E6 must all be `PASS`. Anything else is per-target: run the block for whoeveryou are writing to first, and disclose the skips in the email rather than leaving them out.

In [ ]:
ev = load_evidence()order = [f"E{i}" for i in range(1, 16)]print(f"{'id':<5}{'status':<8}{'experiment':<34}headline")print("-" * 110)for eid in order:    r = ev.get(eid)    if not r:        print(f"{eid:<5}{'NOT RUN':<8}")        continue    print(f"{eid:<5}{r['status']:<8}{r['title']:<34}{r['headline'][:60]}")missing = [e for e in sorted(BLOCKING) if ev.get(e, {}).get("status") != "PASS"]print("\n" + "=" * 110)if missing:    print(f"NO-GO -- blocking experiments not passing: {', '.join(missing)}")else:    skipped = [e for e in order if ev.get(e, {}).get("status") == "SKIP"]    print("GO -- all blocking experiments pass.")    if skipped:        print(f"Disclose these as untested in the email: {', '.join(skipped)}")print("=" * 110)print(f"\nevidence file: {EVIDENCE}")